# Titanic Dataset: EDA & Missing Value Imputation


This notebook coverS:
1. Introduction to Exploratory Data Analysis (EDA)
2. Measures of Central Tendency — Mean, Median, Mode
3. Measures of Dispersion — Standard Deviation, Variance, Range, IQR
4. Missing Value Imputation — Constant, Mean, Median, Mode

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# Option 1: Load from seaborn's built-in dataset
df = sns.load_dataset("titanic")

# Option 2 (alternative): Load from a raw CSV URL
# df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

df.head()



### Section 1: Introduction to Exploratory Data Analysis


## Q1. First Look at the Data
Load the Titanic dataset and display its shape, column names, and data types. What are the first 10 rows of the dataset?


## Q2. Data Structure Audit
Classify each column in the dataset as numerical, ordinal, or nominal categorical. Justify your classification for `Pclass`, `Sex`, and `Fare`.


## Q3. Univariate Exploration
Plot histograms and boxplots for `Age` and `Fare`. What do these plots tell you about the shape and skewness of each distribution?


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Age histogram
axes[0, 0].hist(df['age'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Age Histogram')
axes[0, 0].set_xlabel('Age')

# Age boxplot
axes[0, 1].boxplot(df['age'].dropna())
axes[0, 1].set_title('Age Boxplot')

# Fare histogram
axes[1, 0].hist(df['fare'].dropna(), bins=30, color='darkorange', edgecolor='black')
axes[1, 0].set_title('Fare Histogram')
axes[1, 0].set_xlabel('Fare')

# Fare boxplot
axes[1, 1].boxplot(df['fare'].dropna())
axes[1, 1].set_title('Fare Boxplot')

plt.tight_layout()
plt.show()


## Q4. Bivariate Exploration — Survival vs Features
Create bar plots showing survival rate by `Sex`, `Pclass`, and `Embarked`. Which of these features appears most strongly associated with survival, and why?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival rate by Sex
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')

# Survival rate by Pclass
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[1], color='cornflowerblue')
axes[1].set_title('Survival Rate by Pclass')
axes[1].set_ylabel('Survival Rate')

# Survival rate by Embarked
df.groupby('embarked')['survived'].mean().plot(kind='bar', ax=axes[2], color='indianred')
axes[2].set_title('Survival Rate by Embarked')
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()


## Q5. Correlation Analysis
Generate a correlation heatmap for the numeric features against `Survived`. Which features show the strongest correlation, and are there any signs of multicollinearity?


In [ ]:
# Hint: select numeric columns only, use df.corr(), then sns.heatmap()
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()

corr['survived'].sort_values(ascending=False)



### Section 2: Measures of Central Tendency


## Q6. Mean Age by Passenger Class
What is the mean age of passengers in each `Pclass`? What does this tell you about the age profile of each class?


## Q7. Median Fare by Embarkation Port
What is the median `Fare` for passengers who embarked at each port (`Embarked`)? Why might median be more appropriate than mean here?


## Q8. Mode of Categorical Columns
What is the mode of the `Embarked` and `Pclass` columns? How could this value be useful later in the pipeline?


## Q9. Mean vs Median Comparison on Fare
Compare the mean and median of `Fare`. Why do they differ, and what does this difference indicate about the distribution?


## Q10. Central Tendency by Survival Outcome
How do the mean `Age` and mean `Fare` differ between survivors and non-survivors? What might explain this difference?


---
# Section 3: Measures of Dispersion


## Q11. Standard Deviation of Age and Fare
What is the standard deviation of `Age` and `Fare`? What does a higher standard deviation in `Fare` suggest about ticket pricing?


## Q12. Variance and Its Relationship to Std Dev
Manually calculate the variance of `Fare` and verify that its square root equals the standard deviation. Why is standard deviation generally preferred for interpretation over variance?


In [ ]:
# 1. Manually compute variance: sum((x - mean)^2) / (n - 1)
fare = df['fare'].dropna()
n = len(fare)
mean_fare = fare.mean()
manual_var = ((fare - mean_fare) ** 2).sum() / (n - 1)
print("Manual variance:", manual_var)

# 2. Take the square root and compare it to df['fare'].std()
manual_std = np.sqrt(manual_var)
print("Manual std (sqrt of manual variance):", manual_std)
print("df['fare'].std():", df['fare'].std())

# 3. Also compare against df['fare'].var()
print("df['fare'].var():", df['fare'].var())


## Q13. Range Calculation
What is the range of `Age` and `Fare` in the dataset? Do the minimum or maximum values suggest any data quality issues?


In [ ]:
# Hint: max() - min() for both columns; also print the actual min and max values
age_range = df['age'].max() - df['age'].min()
fare_range = df['fare'].max() - df['fare'].min()

print("Age -> min:", df['age'].min(), "max:", df['age'].max(), "range:", age_range)
print("Fare -> min:", df['fare'].min(), "max:", df['fare'].max(), "range:", fare_range)


## Q14. Interquartile Range (IQR) and Outlier Detection
Calculate Q1, Q3, and the IQR for `Fare`. Using the 1.5×IQR rule, which values would you flag as outliers?


In [ ]:
# 1. Compute Q1 (25th percentile) and Q3 (75th percentile) of 'fare'
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)

# 2. Compute IQR = Q3 - Q1
IQR = Q3 - Q1
print("Q1:", Q1, "Q3:", Q3, "IQR:", IQR)

# 3. Compute lower and upper bounds: Q1 - 1.5*IQR, Q3 + 1.5*IQR
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("Lower bound:", lower_bound, "Upper bound:", upper_bound)

# 4. Filter and display rows considered outliers
outliers = df[(df['fare'] < lower_bound) | (df['fare'] > upper_bound)]
print("Number of outliers:", len(outliers))
outliers.head(10)


## Q15. Dispersion Comparison Across Groups
Compare the standard deviation and IQR of `Fare` across the three passenger classes. What does this reveal about fare variability within each class?


In [22]:
dispersion = df.groupby('pclass')['fare'].agg(
    std='std',
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)
)
dispersion


,std,iqr
pclass,,
1,78.380373,62.57605
2,13.417399,13.00000
3,11.778142,7.75000


---
# Section 4: Missing Value Imputation


## Q16. Missing Value Audit
What percentage of values are missing in the `Age`, `Cabin`/`Deck`, and `Embarked` columns? Based on this, which columns would you consider dropping versus imputing?


## Q17. Constant Imputation
Impute the missing values in `Cabin`/`Deck` with a constant value such as `"Unknown"`. Why might this be more appropriate than deleting the column entirely?


## Q18. Mean Imputation for Age
Impute the missing `Age` values using the mean. How does this change the shape and variance of the `Age` distribution compared to the original?


In [ ]:


# 4. Plot histograms of original vs mean-imputed age side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['age'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Original Age Distribution')
axes[1].hist(df_mean['age'], bins=30, color='darkorange', edgecolor='black')
axes[1].set_title('Mean-Imputed Age Distribution')
plt.tight_layout()
plt.show()

# 5. Compare df['age'].var() vs df_mean['age'].var()
print("Original age variance:", df['age'].var())
print("Mean-imputed age variance:", df_mean['age'].var())


## Q19. Median Imputation for Age
Impute the missing `Age` values using the median instead, and compare the resulting distribution to the mean-imputed version from Q18. Which method better preserves the original distribution, and why?


In [ ]:
# 4. Plot histograms of original vs mean-imputed vs median-imputed age together
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['age'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Original Age')
axes[1].hist(df_mean['age'], bins=30, color='darkorange', edgecolor='black')
axes[1].set_title('Mean-Imputed Age')
axes[2].hist(df_median['age'], bins=30, color='seagreen', edgecolor='black')
axes[2].set_title('Median-Imputed Age')
plt.tight_layout()
plt.show()

# 5. Compare variance and skewness across all three versions
print("Original      -> variance:", df['age'].var(), " skew:", df['age'].skew())
print("Mean-imputed  -> variance:", df_mean['age'].var(), " skew:", df_mean['age'].skew())
print("Median-imputed-> variance:", df_median['age'].var(), " skew:", df_median['age'].skew())


## Q20. Mode Imputation + Grouped Median Imputation Challenge
Impute the missing `Embarked` values using the mode. Then, instead of using a single global median for `Age`, impute missing ages using the **median grouped by `Pclass` and `Sex`**. How does this grouped approach compare to the simple median imputation in Q19?


In [ ]:
# 1. Find the mode of 'embarked'


In [ ]:
# 2. Fill missing values in df['embarked'] with that mode


In [ ]:
# 1. Make a copy: df_grouped = df.copy()


In [ ]:
# 2. Compute median age grouped by ['pclass', 'sex']


In [ ]:
# 3. Use groupby(...).transform() to fill missing 'age' values with the
#    group-specific median (fall back to global median if a group has no data)
global_median = df_grouped['age'].median()
group_median_transform = df_grouped.groupby(['pclass', 'sex'])['age'].transform('median')
df_grouped['age'] = df_grouped['age'].fillna(group_median_transform)
df_grouped['age'] = df_grouped['age'].fillna(global_median)  # fallback for empty groups


In [ ]:
# 4. Compare the resulting distribution (mean, std, histogram) to the
#    single-median version from Q19
print("\nMedian-imputed (global) -> mean:", df_median['age'].mean(), " std:", df_median['age'].std())
print("Grouped-median imputed  -> mean:", df_grouped['age'].mean(), " std:", df_grouped['age'].std())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_median['age'], bins=30, color='seagreen', edgecolor='black')
axes[0].set_title('Global Median-Imputed Age')
axes[1].hist(df_grouped['age'], bins=30, color='purple', edgecolor='black')
axes[1].set_title('Grouped Median-Imputed Age')
plt.tight_layout()
plt.show()
